# Intelligent 3D Drone Navigation — training on Kaggle

**Run this with `Save & Run All (Commit)`, not interactively.** A committed
run keeps going on Kaggle's servers after you close the laptop.

Before the first run:

1. Settings -> Accelerator -> **GPU T4 x2** (or P100)
2. Settings -> Internet -> **On** (needs phone verification; required for
   `git clone`)
3. Add Data -> the `zurich-urban-micro-aerial-vehicle` dataset

**This notebook's very first real run should be with `INSPECT_ONLY = True`
below.** The exact CSV column names in the Zurich MAV dataset are not
confirmed from public docs alone -- inspect mode just prints real
headers/dtypes/sampling rates and parses nothing. Check that output,
adjust `src/data/prepare.py`'s `COLUMN_HINTS` locally if needed, push,
then flip `INSPECT_ONLY = False` and re-run for real.

In [ ]:
# ------------------------------- CONFIG ------------------------------- #
REPO_URL = "https://github.com/shapokok/drone.git"  # <-- yours, push the repo first
CODE_INPUT = None  # or "/kaggle/input/drone-code" for a private-repo workaround

# RAW_INPUT is auto-detected in the next cell (mount path varies between
# UI-created and API-pushed kernels) -- nothing to set here.
PROCESSED_DIR = "/kaggle/working/drone/data/processed"

INSPECT_ONLY = True   # <-- flip to False only after checking the inspect output

MODELS = ["fusion_transformer", "fusion_lstm"]
ABLATIONS = ["full", "no_cross_attn", "imu_only", "gps_only"]  # fusion_transformer only
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 40
BATCH_SIZE = 64
WINDOW = 192

EVAL_OUTAGE_RATES = [0.0, 0.1, 0.3, 0.5, 0.7]

# Stop launching new runs after this many hours so the notebook always
# gets to save its output before Kaggle's 12-hour cutoff.
DEADLINE_HOURS = 10.5

In [ ]:
import os, subprocess, sys, time, shutil, pathlib
T_START = time.time()

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "drone"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if CODE_INPUT:
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(CODE_INPUT, REPO)
elif not REPO.exists():
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")

os.chdir(REPO)
print("cwd:", os.getcwd())

# The dataset mount point under /kaggle/input isn't always the plain
# <dataset-slug> path -- an API-pushed kernel run mounted it one level
# deeper, at /kaggle/input/datasets/<owner>/<slug>, instead of the UI
# convention. Auto-detect by locating RawAccel.csv rather than assume
# either layout, so this works regardless of how the kernel was created.
RAW_INPUT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "RawAccel.csv" in files:
        RAW_INPUT = root
        break
if RAW_INPUT is None:
    print("raw input: MISSING -- add the zurich-urban-micro-aerial-vehicle dataset as input")
else:
    print("raw input:", RAW_INPUT)
    print(sorted(os.listdir(RAW_INPUT)))

## Step 0 — inspect the real CSV schema

Prints headers/dtypes/sampling rates for every CSV under `RAW_INPUT`.
No parsing assumptions are applied here. Compare against
`src/data/prepare.py`'s `COLUMN_HINTS` before trusting anything below.

In [ ]:
sh(f"python src/data/prepare.py --raw {RAW_INPUT} --inspect")

if INSPECT_ONLY:
    raise SystemExit("INSPECT_ONLY=True -- check the output above, fix COLUMN_HINTS "
                      "locally + push if needed, then flip INSPECT_ONLY=False and re-run.")

## Step 1 — prepare (parses the raw CSVs into synced npy arrays)

In [ ]:
sh(f"python src/data/prepare.py --raw {RAW_INPUT} --out {PROCESSED_DIR}")

## Step 2 — environment capture

Written into the notebook output so a reader can reconstruct the exact
setup (paper reproducibility).

In [ ]:
import json, platform, subprocess, sys, torch, pathlib

OUT = pathlib.Path("/kaggle/working/drone/outputs")
OUT.mkdir(parents=True, exist_ok=True)

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(OUT / "requirements_frozen.txt").write_text(freeze)

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"),
}
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## Step 3 — resume from a previous run

If this notebook's own prior output was added as an input dataset,
`results.csv` and `checkpoints/` are seeded from it so `train.py`/
`evaluate.py` skip whatever's already done.

In [ ]:
RESULTS_CSV = str(OUT / "results.csv")

prev = None
for p in pathlib.Path("/kaggle/input").glob("*/outputs"):
    if (p / "results.csv").exists():
        prev = p
        break

if prev:
    print("resuming from", prev)
    shutil.copy(prev / "results.csv", RESULTS_CSV)
    if (prev / "checkpoints").exists():
        shutil.copytree(prev / "checkpoints", OUT / "checkpoints", dirs_exist_ok=True)
    import pandas as pd
    print(len(pd.read_csv(RESULTS_CSV)), "rows already logged")
else:
    print("fresh start")

## Step 4 — train FusionTransformer / FusionLSTM

All ablation arms and seeds for the full protocol. Each finished
(model, ablation, seed) triple appends one row to `results.csv` and is
skipped on re-run.

In [ ]:
def run_train(model, ablation, seed, elapsed_guard=True):
    if elapsed_guard and (time.time() - T_START) / 3600 > DEADLINE_HOURS:
        print(f"\n== deadline reached, stopping so output gets saved. "
              f"Re-run with this output added as input to continue. ==")
        return False
    cmd = [
        sys.executable, "src/train.py",
        "--model", model, "--ablation", ablation, "--seed", str(seed),
        "--epochs", str(EPOCHS), "--batch_size", str(BATCH_SIZE), "--window", str(WINDOW),
        "--processed_dir", PROCESSED_DIR, "--out_dir", str(OUT),
        "--results_csv", RESULTS_CSV,
    ]
    print("\n" + "=" * 70)
    print(" ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print(f"!! {model}/{ablation} seed {seed} exited with {r.returncode}")
    return True

stop = False
for seed in SEEDS:
    for model in MODELS:
        if stop:
            break
        arms = ABLATIONS if model == "fusion_transformer" else ["full"]
        for ablation in arms:
            if not run_train(model, ablation, seed):
                stop = True
                break
    if stop:
        break

print(f"\ntotal {(time.time() - T_START)/3600:.2f} h")

## Step 5 — GPS-denied outage sweep (EKF baseline + trained models)

Same synthetic outage masks are reused across models (see `evaluate.py`),
so Table 2 is an apples-to-apples comparison.

In [ ]:
def run_eval(model, ablation, seed):
    cmd = [
        sys.executable, "src/evaluate.py",
        "--model", model, "--ablation", ablation, "--seed", str(seed),
        "--outage_rates", *[str(r) for r in EVAL_OUTAGE_RATES],
        "--window", str(WINDOW),
        "--processed_dir", PROCESSED_DIR, "--out_dir", str(OUT),
        "--results_csv", RESULTS_CSV,
    ]
    print("\n" + "=" * 70)
    print(" ".join(cmd))
    subprocess.run(cmd)

# EKF baseline: full + no-bias ablation, no seed dependence in the filter
# itself but keep seed=0 as the outage-mask/logging key
run_eval("ekf", "full", 0)
run_eval("ekf", "no_bias", 0)

# trained models, all seeds that finished training above
import pandas as pd
done = pd.read_csv(RESULTS_CSV) if pathlib.Path(RESULTS_CSV).exists() else pd.DataFrame()
trained = done[done.epochs_run.notna()][["model", "ablation", "seed"]].drop_duplicates() \
    if "epochs_run" in done.columns else pd.DataFrame()

for _, row in trained.iterrows():
    run_eval(row.model, row.ablation, int(row.seed))

## Step 6 — XAI attention extraction (Figure 3)

In [ ]:
cmd = [sys.executable, "src/xai.py",
       "--processed_dir", PROCESSED_DIR, "--out_dir", str(OUT),
       "--seed", "0", "--outage_rate", "0.3", "--window", str(WINDOW)]
print(" ".join(cmd))
subprocess.run(cmd)

## Step 7 — build tables + figures

In [ ]:
sys.path.insert(0, "src")
from report import build_all

tables = build_all(RESULTS_CSV, str(OUT), str(OUT / "predictions"))

shutil.copytree(OUT, WORK / "outputs", dirs_exist_ok=True)
print("saved to /kaggle/working/outputs")